In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, average_precision_score
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from tqdm import tqdm
import time
from sklearn.model_selection import KFold
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from tqdm import tqdm
import time

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from tqdm import tqdm
import time
# Custom CNN model for multi-label classification
class PlantCNN(nn.Module):
    def __init__(self, num_classes):
        super(PlantCNN, self).__init__()
        
        # Conv layers
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        
        # Batch normalization
        self.bn1 = nn.BatchNorm2d(64)
        self.bn2 = nn.BatchNorm2d(128)
        self.bn3 = nn.BatchNorm2d(256)
        self.bn4 = nn.BatchNorm2d(512)
        
        # Pooling
        self.pool = nn.MaxPool2d(2, 2)
        
        # Dropout
        self.dropout = nn.Dropout(0.5)
        
        # Fully connected layers
        self.fc1 = nn.Linear(512 * 14 * 14, 1024)
        self.fc2 = nn.Linear(1024, num_classes)
        
    def forward(self, x):
        # Conv block 1
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        
        # Conv block 2
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        
        # Conv block 3
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        
        # Conv block 4
        x = self.pool(F.relu(self.bn4(self.conv4(x))))
        
        # Flatten
        x = x.view(-1, 512 * 14 * 14)
        
        # FC layers
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        # No activation - we'll use BCEWithLogitsLoss which combines sigmoid and BCE
        return x

# Create a deeper CNN model with residual connections
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out

class PlantResNet(nn.Module):
    def __init__(self, num_classes):
        super(PlantResNet, self).__init__()
        self.in_channels = 64
        
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        # Residual blocks
        self.layer1 = self._make_layer(64, 2, stride=1)
        self.layer2 = self._make_layer(128, 2, stride=2)
        self.layer3 = self._make_layer(256, 2, stride=2)
        self.layer4 = self._make_layer(512, 2, stride=2)
        
        # Global average pooling and fully connected layer
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)
        
    def _make_layer(self, out_channels, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for stride in strides:
            layers.append(ResidualBlock(self.in_channels, out_channels, stride))
            self.in_channels = out_channels
        return nn.Sequential(*layers)
    
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.maxpool(out)
        
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out

# Data preparation functions
def prepare_data(df, img_size=224):
    
    # Create class mapping
    classes = sorted(df['class'].unique())
    class_map = {cls: i for i, cls in enumerate(classes)}
    
    # Define transforms
    train_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(30),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Split data into train, validation, and test sets
    # First get unique image IDs
    image_ids = df['Image_ID'].unique()
    train_ids, temp_ids = train_test_split(image_ids, test_size=0.3, random_state=42)
    val_ids, test_ids = train_test_split(temp_ids, test_size=0.5, random_state=42)
    
    # Filter dataframe based on image IDs
    train_df = df[df['Image_ID'].isin(train_ids)]
    val_df = df[df['Image_ID'].isin(val_ids)]
    test_df = df[df['Image_ID'].isin(test_ids)]
    
    # Create datasets
    train_dataset = PlantDataset(train_df, class_map, transform=train_transform)
    val_dataset = PlantDataset(val_df, class_map, transform=val_transform)
    test_dataset = PlantDataset(test_df, class_map, transform=val_transform)
    
    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)
    
    return train_loader, val_loader, test_loader, class_map, classes

# First, let's add the function to calculate mean average precision (mAP)
def calculate_map(y_true, y_pred_probs):
    """
    Calculate Mean Average Precision (mAP) for multi-label classification.
    
    Args:
        y_true: Ground truth binary labels (n_samples, n_classes)
        y_pred_probs: Predicted probabilities (n_samples, n_classes)
    
    Returns:
        mAP score and per-class AP scores
    """
    # Calculate AP for each class
    ap_scores = []
    n_classes = y_true.shape[1]
    
    for i in range(n_classes):
        # Handle classes with no positive samples
        if np.sum(y_true[:, i]) > 0:
            ap = average_precision_score(y_true[:, i], y_pred_probs[:, i])
        else:
            ap = 0.0
        ap_scores.append(ap)
    
    # Calculate mean AP
    mean_ap = np.mean(ap_scores)
    
    return mean_ap, ap_scores

# Modify the train_model function to include mAP and k-fold validation
def train_model(model, train_loader, val_loader, class_names, num_epochs=20, 
                learning_rate=0.001, batch_size=32, use_kfold=False, k_folds=5, full_dataset=None):
    """
    Train the CNN model with mAP metric and optional k-fold cross-validation.
    
    Args:
        model: PyTorch model
        train_loader: Training data loader
        val_loader: Validation data loader
        class_names: List of class names
        num_epochs: Number of training epochs
        learning_rate: Learning rate
        batch_size: Batch size used for training
        use_kfold: Whether to use k-fold cross-validation
        k_folds: Number of folds for cross-validation
        full_dataset: Full dataset for k-fold (required if use_kfold is True)
    
    Returns:
        Trained model and training history
    """
    # Use CUDA if available
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # If k-fold validation is requested
    if use_kfold:
        if full_dataset is None:
            raise ValueError("Full dataset must be provided for k-fold cross-validation")
        
        print(f"Starting {k_folds}-fold cross-validation")
        kfold = KFold(n_splits=k_folds, shuffle=True, random_state=42)
        fold_results = []
        
        for fold, (train_ids, val_ids) in enumerate(kfold.split(full_dataset)):
            print(f"FOLD {fold+1}/{k_folds}")
            print("-" * 60)
            
            # Create data loaders for this fold
            train_subsampler = torch.utils.data.SubsetRandomSampler(train_ids)
            val_subsampler = torch.utils.data.SubsetRandomSampler(val_ids)
            
            fold_train_loader = torch.utils.data.DataLoader(
                full_dataset, 
                batch_size=batch_size,
                sampler=train_subsampler,
                num_workers=4
            )
            
            fold_val_loader = torch.utils.data.DataLoader(
                full_dataset,
                batch_size=batch_size,
                sampler=val_subsampler,
                num_workers=4
            )
            
            # Train model on this fold
            fold_model = type(model)(len(class_names)).to(device)  # Create a new model instance
            fold_model, fold_history = _train_single_fold(
                fold_model, 
                fold_train_loader, 
                fold_val_loader, 
                class_names, 
                num_epochs, 
                learning_rate, 
                batch_size,
                fold=fold+1
            )
            
            # Evaluate model on validation set for this fold
            fold_metrics = evaluate_model(fold_model, fold_val_loader, class_names)
            fold_results.append({
                'fold': fold+1,
                'model': fold_model,
                'history': fold_history,
                'mAP': fold_metrics['mean_ap'],
                'metrics': fold_metrics
            })
            
            print(f"Fold {fold+1} completed. Validation mAP: {fold_metrics['mean_ap']:.4f}")
            print("-" * 60)
        
        # Find best fold based on mAP
        best_fold = max(fold_results, key=lambda x: x['mAP'])
        print(f"Cross-validation completed.")
        print(f"Best fold: {best_fold['fold']} with mAP: {best_fold['mAP']:.4f}")
        
        # Calculate average metrics across all folds
        avg_map = np.mean([fold['mAP'] for fold in fold_results])
        avg_f1 = np.mean([fold['metrics']['macro_f1'] for fold in fold_results])
        
        print(f"Average mAP across all folds: {avg_map:.4f}")
        print(f"Average F1 across all folds: {avg_f1:.4f}")
        
        return best_fold['model'], best_fold['history']
    
    else:
        # Standard training without k-fold
        model = model.to(device)
        return _train_single_fold(model, train_loader, val_loader, class_names, 
                                  num_epochs, learning_rate, batch_size)

# Helper function for training a single fold (or standard training)
def _train_single_fold(model, train_loader, val_loader, class_names, num_epochs, 
                       learning_rate, batch_size, fold=None):
    """
    Train the model on a single fold or standard training.
    Helper function to avoid code duplication.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    
    # Loss function and optimizer
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3)
    
    # Training history
    history = {
        'train_loss': [],
        'val_loss': [],
        'val_f1_macro': [],
        'val_map': [],  # Track mAP scores
        'train_batch_losses': [],
        'class_f1_scores': [],
        'class_ap_scores': [],  # Track per-class AP scores
        'learning_rates': []
    }
    
    fold_str = f" (Fold {fold})" if fold is not None else ""
    best_val_map = 0.0
    best_model_path = f'best_plant_cnn_model{fold_str}.pth'
    
    # Log training start
    print(f"Starting training at {time.strftime('%Y-%m-%d %H:%M:%S')}{fold_str}")
    print(f"Model architecture:\n{model}")
    print(f"Number of trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")
    print(f"Optimizer: {optimizer}")
    print(f"Learning rate: {learning_rate}")
    print(f"Batch size: {batch_size}")
    print(f"Classes: {class_names}")
    print("-" * 60)
    
    for epoch in range(num_epochs):
        epoch_start_time = time.time()
        
        # Training phase
        model.train()
        train_loss = 0.0
        batch_losses = []
        
        print(f"Epoch {epoch+1}/{num_epochs}{fold_str} - Training Phase:")
        progress_bar = tqdm(train_loader, desc=f"Training Epoch {epoch+1}")
        
        for batch_idx, (images, targets) in enumerate(progress_bar):
            batch_start_time = time.time()
            images, targets = images.to(device), targets.to(device)
            
            # Zero the parameter gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, targets)
            
            # Backward pass and optimize
            loss.backward()
            optimizer.step()
            
            # Calculate batch statistics
            batch_loss = loss.item()
            train_loss += batch_loss * images.size(0)
            batch_losses.append(batch_loss)
            
            # Detailed batch logging (every 10 batches)
            if (batch_idx + 1) % 10 == 0:
                current_lr = optimizer.param_groups[0]['lr']
                
                # Calculate batch accuracy
                batch_preds = torch.sigmoid(outputs) > 0.5
                batch_correct = (batch_preds == targets).float().sum().item()
                batch_total = targets.numel()
                batch_accuracy = batch_correct / batch_total * 100
                
                print(f"  Batch {batch_idx+1}/{len(train_loader)}: "
                      f"Loss: {batch_loss:.4f}, "
                      f"Accuracy: {batch_accuracy:.2f}%, "
                      f"LR: {current_lr:.6f}, "
                      f"Time: {time.time() - batch_start_time:.2f}s")
                
                # Log a sample of predictions from this batch
                if batch_idx % 50 == 0:
                    sample_idx = 0  # First sample in the batch
                    print(f"  Sample prediction from batch {batch_idx+1}:")
                    print(f"    Input shape: {images[sample_idx].shape}")
                    
                    sample_output = outputs[sample_idx].cpu().detach().numpy()
                    sample_probs = 1 / (1 + np.exp(-sample_output))  # sigmoid
                    sample_preds = sample_probs > 0.5
                    sample_targets = targets[sample_idx].cpu().numpy()
                    
                    for i, (cls_name, pred, prob, target) in enumerate(zip(class_names, sample_preds, sample_probs, sample_targets)):
                        print(f"    Class '{cls_name}': Predicted: {pred} (Prob: {prob:.4f}), True: {target}")
            
            # Update progress bar
            progress_bar.set_postfix(loss=batch_loss)
        
        # Calculate training loss for the epoch
        train_loss = train_loss / len(train_loader.dataset)
        
        # Log end of training phase
        print(f"  Training phase completed in {time.time() - epoch_start_time:.2f}s")
        print(f"  Average training loss: {train_loss:.4f}")
        print("-" * 40)
        
        # Validation phase
        print(f"Epoch {epoch+1}/{num_epochs}{fold_str} - Validation Phase:")
        model.eval()
        val_loss = 0.0
        all_targets = []
        all_preds = []
        all_probs = []
        
        with torch.no_grad():
            progress_bar = tqdm(val_loader, desc=f"Validation Epoch {epoch+1}")
            
            for batch_idx, (images, targets) in enumerate(progress_bar):
                images, targets = images.to(device), targets.to(device)
                
                # Forward pass
                outputs = model(images)
                loss = criterion(outputs, targets)
                
                val_loss += loss.item() * images.size(0)
                
                # Convert outputs to probabilities and predictions
                probs = torch.sigmoid(outputs)
                preds = probs > 0.5
                
                # Store targets, predictions, and probabilities for metrics calculation
                all_targets.append(targets.cpu().numpy())
                all_preds.append(preds.cpu().numpy())
                all_probs.append(probs.cpu().numpy())
                
                # Log sample predictions (every 5 batches in validation)
                if batch_idx % 5 == 0:
                    sample_idx = 0
                    print(f"\n  Validation sample predictions (Batch {batch_idx+1}, Sample {sample_idx}):")
                    sample_output = outputs[sample_idx].cpu().numpy()
                    sample_probs = probs[sample_idx].cpu().numpy()
                    sample_preds = preds[sample_idx].cpu().numpy()
                    sample_targets = targets[sample_idx].cpu().numpy()
                    
                    print("  Class                   | Prediction | Probability | True Label")
                    print("  -----------------------|------------|-------------|------------")
                    for i, cls_name in enumerate(class_names):
                        status = "✓" if sample_preds[i] == sample_targets[i] else "✗"
                        print(f"  {cls_name.ljust(22)} | {int(sample_preds[i])}          | {sample_probs[i]:.4f}      | {int(sample_targets[i])}          {status}")
                    
                # Update progress bar with current loss
                progress_bar.set_postfix(loss=loss.item())
        
        # Calculate validation metrics
        val_loss = val_loss / len(val_loader.dataset)
        all_targets = np.vstack(all_targets)
        all_preds = np.vstack(all_preds)
        all_probs = np.vstack(all_probs)
        
        # Calculate mAP and per-class AP
        mean_ap, ap_scores = calculate_map(all_targets, all_probs)
        
        # Calculate per-class metrics (for compatibility with original code)
        precision, recall, f1, support = precision_recall_fscore_support(
            all_targets, all_preds, average=None, zero_division=0
        )
        
        # Calculate macro F1 score
        macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
            all_targets, all_preds, average='macro', zero_division=0
        )
        
        # Store per-class scores in history
        class_f1_dict = {class_names[i]: f1[i] for i in range(len(class_names))}
        class_ap_dict = {class_names[i]: ap_scores[i] for i in range(len(class_names))}
        history['class_f1_scores'].append(class_f1_dict)
        history['class_ap_scores'].append(class_ap_dict)
        
        # Update learning rate
        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step(val_loss)
        new_lr = optimizer.param_groups[0]['lr']
        
        # Store history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_f1_macro'].append(macro_f1)
        history['val_map'].append(mean_ap)  # Store mAP
        history['train_batch_losses'].extend(batch_losses)
        history['learning_rates'].append(current_lr)
        
        # Print detailed validation results
        print("\nValidation Results:")
        print(f"  Val Loss: {val_loss:.4f}")
        print(f"  Val Mean AP: {mean_ap:.4f}")  # Display mAP
        print(f"  Val Macro F1: {macro_f1:.4f}")
        print(f"  Val Macro Precision: {macro_precision:.4f}")
        print(f"  Val Macro Recall: {macro_recall:.4f}")
        
        # Print per-class metrics
        print("\nPer-class metrics:")
        print("  Class                   | AP        | Precision | Recall   | F1 Score | Support")
        print("  -----------------------|-----------|-----------|----------|----------|--------")
        for i, cls_name in enumerate(class_names):
            print(f"  {cls_name.ljust(22)} | {ap_scores[i]:.4f}    | {precision[i]:.4f}    | {recall[i]:.4f}   | {f1[i]:.4f}    | {support[i]}")
        
        # Log learning rate changes
        if new_lr != current_lr:
            print(f"\nLearning rate decreased from {current_lr:.6f} to {new_lr:.6f}")
        
        # Save best model based on mAP (instead of F1)
        if mean_ap > best_val_map:
            best_val_map = mean_ap
            torch.save(model.state_dict(), best_model_path)
            print(f"\n✓ Saved best model with mAP: {mean_ap:.4f}")
        
        # Calculate confusion matrix
        if epoch % 5 == 0 or epoch == num_epochs - 1:
            print("\nGenerating confusion matrices for each class...")
            
            for i, cls_name in enumerate(class_names):
                tn, fp, fn, tp = confusion_matrix(all_targets[:, i], all_preds[:, i]).ravel()
                print(f"  Confusion Matrix for '{cls_name}':")
                print(f"    True Positives: {tp}")
                print(f"    False Positives: {fp}")
                print(f"    True Negatives: {tn}")
                print(f"    False Negatives: {fn}")
        
        # Log epoch summary
        epoch_time = time.time() - epoch_start_time
        print(f"\nEpoch {epoch+1} completed in {epoch_time:.2f}s")
        print(f"Current best validation mAP: {best_val_map:.4f}")
        print("=" * 80)
    
    # Final summary
    print(f"\nTraining completed at {time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Best validation mAP score: {best_val_map:.4f}")
    print(f"Best model saved to: {best_model_path}")
    
    # Load best model
    model.load_state_dict(torch.load(best_model_path))
    
    return model, history

# Update the evaluate_model function to include mAP
def evaluate_model(model, test_loader, class_names):
    """
    Evaluate the trained model on the test set with mAP.
    
    Args:
        model: Trained PyTorch model
        test_loader: Test data loader
        class_names: List of class names
    
    Returns:
        Evaluation metrics
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()
    
    all_targets = []
    all_preds = []
    all_probs = []
    
    with torch.no_grad():
        for images, targets in test_loader:
            images, targets = images.to(device), targets.to(device)
            
            # Forward pass
            outputs = model(images)
            probs = torch.sigmoid(outputs)
            preds = probs > 0.5
            
            # Store targets and predictions
            all_targets.append(targets.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
            all_probs.append(probs.cpu().numpy())
    
    # Concatenate results
    all_targets = np.vstack(all_targets)
    all_preds = np.vstack(all_preds)
    all_probs = np.vstack(all_probs)
    
    # Calculate mAP
    mean_ap, ap_scores = calculate_map(all_targets, all_probs)
    
    # Calculate metrics
    precision, recall, f1, support = precision_recall_fscore_support(
        all_targets, all_preds, average=None, zero_division=0
    )
    
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        all_targets, all_preds, average='macro', zero_division=0
    )
    
    micro_precision, micro_recall, micro_f1, _ = precision_recall_fscore_support(
        all_targets, all_preds, average='micro', zero_division=0
    )
    
    # Create per-class metrics dataframe
    class_metrics = pd.DataFrame({
        'Class': class_names,
        'AP': ap_scores,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'Support': support
    })
    
    # Sort by AP (Average Precision)
    class_metrics = class_metrics.sort_values('AP', ascending=False)
    
    # Print classification report
    print("\nClassification Report (Multi-label):")
    for i, row in class_metrics.iterrows():
        print(f"Class: {row['Class']}")
        print(f"  AP: {row['AP']:.4f}")
        print(f"  Precision: {row['Precision']:.4f}")
        print(f"  Recall: {row['Recall']:.4f}")
        print(f"  F1-Score: {row['F1-Score']:.4f}")
        print(f"  Support: {row['Support']}")
    
    print("\nOverall Metrics:")
    print(f"  Mean AP: {mean_ap:.4f}")
    print(f"  Macro Precision: {macro_precision:.4f}")
    print(f"  Macro Recall: {macro_recall:.4f}")
    print(f"  Macro F1: {macro_f1:.4f}")
    print(f"  Micro Precision: {micro_precision:.4f}")
    print(f"  Micro Recall: {micro_recall:.4f}")
    print(f"  Micro F1: {micro_f1:.4f}")
    
    # Plot metrics
    plt.figure(figsize=(14, 10))
    metrics_plot = pd.melt(
        class_metrics,
        id_vars=['Class'],
        value_vars=['AP', 'Precision', 'Recall', 'F1-Score'],
        var_name='Metric',
        value_name='Value'
    )
    
    sns.barplot(x='Class', y='Value', hue='Metric', data=metrics_plot)
    plt.title('Per-Class Performance Metrics', fontsize=16)
    plt.xticks(rotation=45, ha='right')
    plt.ylim(0, 1.0)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig('per_class_metrics.png', dpi=300)
    
    return {
        'per_class_metrics': class_metrics,
        'mean_ap': mean_ap,
        'ap_scores': ap_scores,
        'macro_precision': macro_precision,
        'macro_recall': macro_recall,
        'macro_f1': macro_f1,
        'micro_precision': micro_precision,
        'micro_recall': micro_recall,
        'micro_f1': micro_f1,
        'all_targets': all_targets,
        'all_preds': all_preds,
        'all_probs': all_probs
    }

# Update the plotting function to include mAP
def plot_training_history(history, class_names):
    """
    Plot training and validation metrics history including mAP.
    
    Args:
        history: Dictionary containing training metrics
        class_names: List of class names
    """
    # Create figure with subplots
    fig, axs = plt.subplots(2, 2, figsize=(15, 10))
    
    # Plot training and validation loss
    axs[0, 0].plot(history['train_loss'], label='Training Loss')
    axs[0, 0].plot(history['val_loss'], label='Validation Loss')
    axs[0, 0].set_xlabel('Epoch')
    axs[0, 0].set_ylabel('Loss')
    axs[0, 0].set_title('Training and Validation Loss')
    axs[0, 0].legend()
    axs[0, 0].grid(True)
    
    # Plot metrics (F1 and mAP)
    axs[0, 1].plot(history['val_f1_macro'], label='Validation Macro F1')
    axs[0, 1].plot(history['val_map'], label='Validation mAP')  # Add mAP plot
    axs[0, 1].set_xlabel('Epoch')
    axs[0, 1].set_ylabel('Score')
    axs[0, 1].set_title('Validation Metrics')
    axs[0, 1].legend()
    axs[0, 1].grid(True)
    
    # Plot learning rate
    axs[1, 0].plot(history['learning_rates'], label='Learning Rate')
    axs[1, 0].set_xlabel('Epoch')
    axs[1, 0].set_ylabel('Learning Rate')
    axs[1, 0].set_title('Learning Rate Over Time')
    axs[1, 0].set_yscale('log')
    axs[1, 0].legend()
    axs[1, 0].grid(True)
    
    # Plot per-class AP scores
    ax = axs[1, 1]
    for i, cls_name in enumerate(class_names):
        ap_scores = [epoch_ap[cls_name] for epoch_ap in history['class_ap_scores']]
        ax.plot(ap_scores, label=cls_name)
    
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Average Precision')
    ax.set_title('Per-class AP Scores')
    ax.legend()
    ax.grid(True)
    
    plt.tight_layout()
    plt.savefig('training_history.png')
    plt.show()


img_size = 224
num_epochs = 20
learning_rate = 0.001
batch_size = 64

# Prepare data
train_loader, val_loader, test_loader, class_map, class_names = prepare_data(train_df, img_size)

# Create the model
model = PlantResNet(num_classes=len(class_names))

# Train with k-fold validation (5 folds)
trained_model, history = train_model(
    model, 
    train_loader, 
    val_loader, 
    class_names,
    num_epochs=2, 
    learning_rate=0.001, 
    batch_size=32,
    use_kfold=True,  # Enable k-fold cross-validation
    k_folds=5,       # Number of folds
    full_dataset=train_df  # Pass your full dataset here
)

# Plot training history
plot_training_history(history, class_names)

# Evaluate on test set
metrics = evaluate_model(trained_model, test_loader, class_names)
